## Workflow guide
The Gold layer converts validated OMOP tables into dashboard-ready outputs: headline data-quality metrics, diagnosis summaries and patient utilization measures.

In [0]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "clinical_portfolio"
BASE = f"{CATALOG}.{SCHEMA}"

person = spark.table(f"{BASE}.person")
visit_occurrence = spark.table(f"{BASE}.visit_occurrence")
condition_occurrence = spark.table(f"{BASE}.condition_occurrence")
observation_period = spark.table(f"{BASE}.observation_period")

print("Silver tables loaded successfully.")

Silver tables loaded successfully.


In [0]:
total_persons = person.count()
total_visits = visit_occurrence.count()
total_conditions = condition_occurrence.count()

conditions_with_visit = (
    condition_occurrence
    .filter(F.col("visit_occurrence_id").isNotNull())
    .count()
)

visit_linkage_pct = round(
    100 * conditions_with_visit / total_conditions, 2
)

quality_metrics = spark.createDataFrame(
    [
        ("total_persons", float(total_persons), "Number of unique patients"),
        ("total_visits", float(total_visits), "Number of clinical encounters"),
        ("total_conditions", float(total_conditions), "Number of mapped condition events"),
        (
            "conditions_linked_to_visit_pct",
            float(visit_linkage_pct),
            "Percentage of conditions linked to a visit"
        )
    ],
    ["metric_name", "metric_value", "metric_description"]
)

display(quality_metrics)

metric_name,metric_value,metric_description
total_persons,6.0,Number of unique patients
total_visits,8.0,Number of clinical encounters
total_conditions,9.0,Number of mapped condition events
conditions_linked_to_visit_pct,100.0,Percentage of conditions linked to a visit


In [0]:
quality_metrics.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_quality_metrics")

display(spark.table(f"{BASE}.gold_quality_metrics"))

metric_name,metric_value,metric_description
total_persons,6.0,Number of unique patients
total_visits,8.0,Number of clinical encounters
total_conditions,9.0,Number of mapped condition events
conditions_linked_to_visit_pct,100.0,Percentage of conditions linked to a visit


### Dashboard quality metrics
The metrics table uses a long format: one metric per row. This is convenient for Power BI cards and keeps metric names, values and definitions together.

In [0]:
condition_summary = (
    condition_occurrence
    .groupBy(
        "condition_concept_id",
        "condition_source_value"
    )
    .agg(
        F.count("*").alias("condition_event_count"),
        F.countDistinct("person_id").alias("unique_patient_count"),
        F.min("condition_start_date").alias("first_recorded_date"),
        F.max("condition_start_date").alias("last_recorded_date")
    )
    .orderBy(F.desc("condition_event_count"))
)

display(condition_summary)

condition_concept_id,condition_source_value,condition_event_count,unique_patient_count,first_recorded_date,last_recorded_date
316866,HTN,4,3,2025-01-04,2025-05-22
201826,T2D,2,2,2025-03-15,2025-05-03
317009,ASTHMA,2,2,2025-02-10,2025-04-19
377821,MIGRAINE,1,1,2025-02-28,2025-02-28


In [0]:
condition_summary.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_condition_summary")

display(spark.table(f"{BASE}.gold_condition_summary"))

condition_concept_id,condition_source_value,condition_event_count,unique_patient_count,first_recorded_date,last_recorded_date
316866,HTN,4,3,2025-01-04,2025-05-22
201826,T2D,2,2,2025-03-15,2025-05-03
317009,ASTHMA,2,2,2025-02-10,2025-04-19
377821,MIGRAINE,1,1,2025-02-28,2025-02-28


### Condition summary
This Gold table separates total events from distinct patients. The distinction matters: repeated events for one patient should not be mistaken for a larger patient cohort.

In [0]:
patient_utilization = (
    person
    .join(
        observation_period,
        on="person_id",
        how="left"
    )
    .join(
        visit_occurrence
        .groupBy("person_id")
        .agg(F.count("*").alias("visit_count")),
        on="person_id",
        how="left"
    )
    .join(
        condition_occurrence
        .groupBy("person_id")
        .agg(F.count("*").alias("condition_count")),
        on="person_id",
        how="left"
    )
    .fillna(
        {
            "visit_count": 0,
            "condition_count": 0
        }
    )
    .select(
        "person_id",
        "year_of_birth",
        "gender_source_value",
        "observation_period_start_date",
        "observation_period_end_date",
        "visit_count",
        "condition_count"
    )
)

display(patient_utilization)

person_id,year_of_birth,gender_source_value,observation_period_start_date,observation_period_end_date,visit_count,condition_count
7419654189246804363,1978,F,2025-01-04,2025-03-17,2,3
4905020018283288165,1986,M,2025-02-10,2025-05-22,2,2
7357811929381004785,1994,F,2025-02-28,2025-02-28,1,1
3215699940703695032,1969,M,2025-04-06,2025-04-08,1,1
3871179191872680772,2001,F,2025-04-19,2025-04-19,1,1
2763232168323479912,1982,M,2025-05-03,2025-05-03,1,1


In [0]:
patient_utilization = patient_utilization.withColumn(
    "observation_days",
    F.datediff(
        F.col("observation_period_end_date"),
        F.col("observation_period_start_date")
    )
)

display(patient_utilization)

person_id,year_of_birth,gender_source_value,observation_period_start_date,observation_period_end_date,visit_count,condition_count,observation_days
7419654189246804363,1978,F,2025-01-04,2025-03-17,2,3,72
4905020018283288165,1986,M,2025-02-10,2025-05-22,2,2,101
7357811929381004785,1994,F,2025-02-28,2025-02-28,1,1,0
3215699940703695032,1969,M,2025-04-06,2025-04-08,1,1,2
3871179191872680772,2001,F,2025-04-19,2025-04-19,1,1,0
2763232168323479912,1982,M,2025-05-03,2025-05-03,1,1,0


In [0]:
patient_utilization.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{BASE}.gold_patient_utilization")

display(spark.table(f"{BASE}.gold_patient_utilization"))

person_id,year_of_birth,gender_source_value,observation_period_start_date,observation_period_end_date,visit_count,condition_count,observation_days
7419654189246804363,1978,F,2025-01-04,2025-03-17,2,3,72
4905020018283288165,1986,M,2025-02-10,2025-05-22,2,2,101
7357811929381004785,1994,F,2025-02-28,2025-02-28,1,1,0
3215699940703695032,1969,M,2025-04-06,2025-04-08,1,1,2
3871179191872680772,2001,F,2025-04-19,2025-04-19,1,1,0
2763232168323479912,1982,M,2025-05-03,2025-05-03,1,1,0


### Patient utilization
The patient-level table combines demographics, observation time, visit count and condition count. It is intentionally derived from Silver tables so every analysis measure is reproducible from validated data.

# Gold quality and utilization metrics

This notebook materializes small, analysis-ready Gold tables. It publishes headline quality metrics, diagnosis summaries and patient utilization measures so they can be queried directly or used in a dashboard.